In [1]:
import sys
import os

# 1. The Environment Guardrail
if 'google.colab' not in sys.modules:
    print("CRITICAL ERROR: This interactive notebook is designed exclusively for Google Colab.")
    print("Running this locally in VS Code will fail due to cloud-specific UI widgets.")
    print("Please open this .ipynb file in Google Colab to execute the Inference Engine.")
    sys.exit(1)

print("Colab environment verified. Initiating Modern Environment Setup (uv)...")

# 2. Install uv and update PATH
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] += ":/root/.cargo/bin"

# 3. Clone or Pull Repository
if not os.path.exists('OilPricePrediction'):
    print("Downloading repository...")
    !git clone -q https://github.com/luissejas/OilPricePrediction.git
    %cd OilPricePrediction
else:
    print("Repository found. Pulling latest code and models from GitHub...")
    %cd OilPricePrediction
    !git pull -q origin main

# 4. Install Dependencies via uv (High Speed)
print("Verifying environment dependencies...")
# --system is used to inject packages into the managed Colab kernel
!uv pip install --system -q -r requirements.txt
!uv pip install --system -q torch xgboost yfinance ipywidgets

print("\n✅ Environment Sync Complete. The cloud machine is armed via uv.")

Colab environment verified. Initiating Modern Environment Setup (uv)...
downloading uv 0.10.12 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Repository found. Pulling latest code and models from GitHub...
/content/OilPricePrediction
Verifying environment dependencies...

✅ Environment Sync Complete. The cloud machine is armed via uv.


In [ ]:
# --- 5. COLAB GPU ACCELERATED TRAINING ---
# Ensure you have selected 'T4 GPU' in Runtime -> Change runtime type before running
print("Initializing Cloud Hardware...")
!uv run python src/models/train.py


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

# --- 1. THE NATIVE ARCHITECTURE IMPORT ---
from src.models.pytorch_model import OilPriceNN
print("Blueprint successfully imported from src.models.pytorch_model.")

# --- 2. THE LIVE DATA PIPELINE ---
def fetch_and_engineer_live_data():
    print("Fetching last 60 days of crude oil data (CL=F)...")
    ticker = yf.Ticker("CL=F")
    history = ticker.history(period="60d")

    df = pd.DataFrame()
    df['price'] = history['Close']

    # Engineer the exact features the model was trained on
    df['price_lag_1'] = df['price'].shift(1)
    df['price_lag_3'] = df['price'].shift(3)
    df['sma_7'] = df['price'].rolling(window=7).mean()
    df['sma_14'] = df['price'].rolling(window=14).mean()
    df['volatility_7'] = df['price'].rolling(window=7).std()

    # Drop the 14-day burn-in buffer
    df = df.dropna()
    return df.tail(30)

try:
    live_df = fetch_and_engineer_live_data()
    available_dates = live_df.index.strftime('%Y-%m-%d').tolist()
    print("Live data successfully engineered. UI is ready.")
except Exception as e:
    print(f"Critical Error fetching live data: {e}")
    available_dates = []

Blueprint successfully imported from src.models.pytorch_model.
Fetching last 60 days of crude oil data (CL=F)...
Live data successfully engineered. UI is ready.


In [ ]:
import datetime

def run_inference(selected_date):
    with output_area:
        clear_output(wait=True)
        print("\n" + "="*60)
        print(f"LIVE MARKET PREDICTION FOR: {selected_date}")
        print("="*60)

        day_data = live_df.loc[selected_date]
        actual_price = day_data['price']

        features = ['price_lag_1', 'price_lag_3', 'sma_7', 'sma_14', 'volatility_7']
        raw_metrics = day_data[features].values.reshape(1, -1)

        print(f"Actual Market Price: ${actual_price:.2f}\n")

        # --- XGBOOST INFERENCE ---
        try:
            # We don't need the class for XGBoost inference, just the native loader
            xgb_model = xgb.XGBRegressor()
            # Note the path: root models folder
            xgb_model.load_model("models/champion_xgboost.json")
            xgb_pred = xgb_model.predict(raw_metrics)[0]
            print(f"[XGBoost] Prediction: ${xgb_pred:.2f} (Error: ${abs(actual_price - xgb_pred):.2f})")
        except Exception as e:
            print(f"XGBoost Error: {e}")

        # --- PYTORCH INFERENCE ---
        try:
            scaler = joblib.load("models/nn_scaler.pkl")
            scaled_metrics = scaler.transform(raw_metrics)
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

            nn_model = OilPriceNN(input_size=5).to(device)
            nn_model.load_state_dict(torch.load("models/champion_nn.pth", map_location=device))
            nn_model.eval()

            with torch.no_grad():
                tensor_in = torch.FloatTensor(scaled_metrics).to(device)
                nn_pred = nn_model(tensor_in).item()
            print(f"[PyTorch] Prediction: ${nn_pred:.2f} (Error: ${abs(actual_price - nn_pred):.2f})")
        except Exception as e:
            print(f"PyTorch Error: {e}")

        print("="*60 + "\n")

# --- UI WITH DATE SLIDER ---
output_area = widgets.Output()
date_slider = widgets.SelectionSlider(
    options=available_dates,
    value=available_dates[-1],
    description='Trading Day:',
    layout={'width': '400px'}
)

predict_button = widgets.Button(description='Run AI Models', button_style='success', icon='bolt')
predict_button.on_click(lambda b: run_inference(date_slider.value))

display(widgets.HBox([date_slider, predict_button]), output_area)